# Step 1: Verify merged_482_cases.csv Against metadata.csv, Then Add a Dominant Rhythm Label Column

This notebook does two things, in order:

**Part A — Verification.** The VitalDB Arrhythmia Database publishes its own `metadata.csv` on PhysioNet, which contains a second, independent copy of each patient's clinical data (the paper's authors pulled it from VitalDB themselves when they built the dataset). We already built `merged_482_cases.csv` by pulling the *same* clinical data live from the VitalDB API. These two sources **should** agree almost perfectly. We check this column-by-column to make sure nothing went wrong in the original merge, and to catch any version drift between the live API and the published snapshot.

**Part B — New feature.** We add a `dominant_rhythm` column to `merged_482_cases.csv`: for each patient, the most frequent *non-normal* `rhythm_label` value across all of that patient's annotated beats (excluding any beats flagged as bad signal quality). This is the "what arrhythmia did this patient actually have" feature the model will use.

**Important finding from data exploration (explained in Part B):** the project context document describes 10 rhythm categories using full names like `"Atrial Fibrillation"`. The *actual* annotation files use a mix of abbreviated codes (`N`, `AFIB/AFL`, `SVTA`, `VT`, `WAP/MAT`, `SND`, `AVB`) and a few full names (`Patterned Atrial Ectopy`, `Patterned Ventricular Ectopy`, `Unclassifiable`, `Noise`). We use the values exactly as they appear in the real files, not the full names from the description, since those are what's actually in the data.

In [ ]:
# pandas: for loading and manipulating tabular data (dataframes)
import pandas as pd
# numpy: for numeric operations, especially handling missing values (NaN)
import numpy as np
# Path: a convenient way to build file paths that works on Windows and other systems
from pathlib import Path
# requests + io: for downloading metadata.csv directly from PhysioNet over HTTPS
import requests
import io

print("Libraries loaded successfully.")

In [ ]:
BASE_DIR = Path("..").resolve()

# Folder you downloaded from PhysioNet — see DATA.md for the exact download step
ANNOTATION_DIR = BASE_DIR / "external_data" / "vitaldb-arrhythmia-database-1.0.0" / "Annotation_Files"

# The CSV created in 01_merge_clinical_lab_data.ipynb (clinical + lab data, 482 rows)
MERGED_CSV = BASE_DIR / "data" / "interim" / "merged_482_cases.csv"

# The arrhythmia database's own metadata file, hosted on PhysioNet (no login required)
METADATA_URL = "https://physionet.org/files/vitaldb-arrhythmia/1.0.0/metadata.csv"

print(f"Annotation folder exists: {ANNOTATION_DIR.exists()}")
print(f"Merged CSV exists: {MERGED_CSV.exists()}")

## Part A: Verify merged_482_cases.csv Against metadata.csv

In [ ]:
# Load our existing merged dataframe from disk
merged = pd.read_csv(MERGED_CSV)

# Download the arrhythmia database's metadata.csv straight from PhysioNet
resp = requests.get(METADATA_URL)
resp.raise_for_status()  # raises an error immediately if the download failed
metadata = pd.read_csv(io.StringIO(resp.text))

print(f"merged_482_cases.csv shape : {merged.shape}")
print(f"metadata.csv shape         : {metadata.shape}")

In [ ]:
# Step 1 of verification: do both files contain exactly the same 482 patients?
# Note the column is called "caseid" in our merged file but "case_id" in metadata.csv
merged_ids = set(merged["caseid"])
metadata_ids = set(metadata["case_id"])

print(f"case_ids in merged but not in metadata: {sorted(merged_ids - metadata_ids)}")
print(f"case_ids in metadata but not in merged: {sorted(metadata_ids - merged_ids)}")
print(f"Case ID sets are identical: {merged_ids == metadata_ids}")

In [ ]:
# Step 2 of verification: for every clinical column that appears in BOTH files,
# check whether the values match for every patient.

# Line up both dataframes by case ID so row N in one corresponds to row N in the other
merged_indexed = merged.set_index("caseid").sort_index()
metadata_indexed = metadata.set_index("case_id").sort_index()
metadata_indexed = metadata_indexed.loc[merged_indexed.index]  # same row order

# Columns that exist in both files (excluding the case ID column itself)
overlap_cols = [c for c in merged_indexed.columns if c in metadata_indexed.columns]
print(f"Comparing {len(overlap_cols)} overlapping columns...\n")

mismatch_report = {}
for col in overlap_cols:
    a = merged_indexed[col]
    # Some metadata.csv columns are read as text (object) instead of numbers,
    # usually because of a non-numeric placeholder value somewhere in the column.
    # Try converting both sides to numbers first; if that fails, compare as text.
    a_num = pd.to_numeric(a, errors="coerce")
    b_num = pd.to_numeric(metadata_indexed[col], errors="coerce")
    if a_num.notna().sum() > 0 and b_num.notna().sum() > 0:
        a_cmp, b_cmp = a_num, b_num
    else:
        a_cmp, b_cmp = a.astype(str), metadata_indexed[col].astype(str)

    both_missing = a_cmp.isna() & b_cmp.isna()   # treat NaN == NaN as a match
    values_equal = (a_cmp == b_cmp) | both_missing
    n_mismatched = (~values_equal).sum()
    if n_mismatched > 0:
        mismatch_report[col] = n_mismatched

print(f"Columns with at least one mismatch: {len(mismatch_report)} / {len(overlap_cols)}")
for col, n in mismatch_report.items():
    print(f"  {col}: {n} mismatched rows")

In [ ]:
# Investigate any remaining mismatches in detail before concluding anything is wrong
for col in mismatch_report:
    a_num = pd.to_numeric(merged_indexed[col], errors="coerce")
    b_num = pd.to_numeric(metadata_indexed[col], errors="coerce")
    mismatched_mask = ~((a_num == b_num) | (a_num.isna() & b_num.isna()))
    print(f"--- {col} ---")
    print(pd.DataFrame({
        "merged_value": merged_indexed.loc[mismatched_mask, col],
        "metadata_value": metadata_indexed.loc[mismatched_mask, col],
    }))
    print()

**Verification result:** all 482 case IDs match exactly between the two sources, and all 73 overlapping clinical columns match exactly **except** `age`, which differs for exactly 2 patients (case 2432 and case 2922). In `metadata.csv`, PhysioNet's published version masks any age above 89 as the text `">89"` — a standard de-identification rule (ages 90+ are considered re-identifiable). Our `merged_482_cases.csv` pulled the **exact** ages (92 and 90) directly from the live VitalDB API, which does not apply that masking. This is expected, documented de-identification behavior, not a data error — our merged file is actually more precise here, and we'll keep using it as-is.

## Part B: Discover the Real Rhythm Label Vocabulary

In [ ]:
# Before writing code that extracts a "dominant rhythm" per patient, we need to know
# exactly what string values can appear in the rhythm_label column. Scan every one
# of the 482 annotation files and collect every unique value, with a count of how
# many files contain each one.
annotation_files = sorted(ANNOTATION_DIR.glob("Annotation_file_*.csv"))
print(f"Found {len(annotation_files)} annotation files")

rhythm_value_counts = {}
for f in annotation_files:
    ann = pd.read_csv(f)
    for val in ann["rhythm_label"].dropna().unique():
        rhythm_value_counts[val] = rhythm_value_counts.get(val, 0) + 1

print("\nAll rhythm_label values seen across the 482 files, sorted by how common they are:")
for val, n_files in sorted(rhythm_value_counts.items(), key=lambda x: -x[1]):
    print(f"  {val!r}: appears in {n_files} files")

**What these codes mean** (mapping the abbreviations to the rhythm names from the project description):

| Code in data | Meaning |
|---|---|
| `N` | Normal Sinus Rhythm |
| `AFIB/AFL` | Atrial Fibrillation / Atrial Flutter |
| `SVTA` | Supraventricular Tachyarrhythmia |
| `VT` | Ventricular Tachyarrhythmia |
| `WAP/MAT` | Wandering Atrial Pacemaker / Multifocal Atrial Tachycardia |
| `SND` | Sinus Node Dysfunction |
| `AVB` | Atrioventricular Block |
| `Patterned Atrial Ectopy` | (already a full name) |
| `Patterned Ventricular Ectopy` | (already a full name) |
| `Unclassifiable` | (already a full name) — excluded from training per mentor |
| `Noise` | (already a full name) — excluded from training per mentor |

We will keep the `dominant_rhythm` column in these exact original codes (not translate them), since that's what's actually in the source data and avoids introducing a translation mistake. The rhythm-filtering step (excluding `Unclassifiable`/`Noise`) comes next, after this notebook.

In [ ]:
def get_dominant_rhythm(case_id):
    """
    For one patient's annotation file, return the most frequently occurring
    NON-NORMAL rhythm_label value (e.g. 'AFIB/AFL', 'VT', 'Noise').

    Why exclude normal beats first: each patient's annotated ~20-minute segment
    is mostly Normal Sinus Rhythm ('N') with a smaller arrhythmia episode embedded
    in it. If we took the single most common rhythm_label value over the WHOLE
    file without excluding 'N' first, we would almost always get 'N' back, which
    defeats the purpose of this feature (capturing what arrhythmia the patient had).

    Returns:
        - the most common non-normal rhythm_label string, if any non-normal beats exist
        - 'N' if every remaining (good-quality) beat in the file is Normal Sinus Rhythm
        - None if the annotation file is missing, or every single beat is flagged
          as bad signal quality (so there is nothing reliable to count)
    """
    ann_path = ANNOTATION_DIR / f"Annotation_file_{case_id}.csv"
    if not ann_path.exists():
        return None

    ann = pd.read_csv(ann_path)

    # Drop rows flagged as bad signal quality — these are noise artifacts, not
    # reliable rhythm readings, and the mentor's instructions say to exclude them
    ann = ann[ann["bad_signal_quality"] == False]
    if ann.empty:
        return None

    # Keep only rows where the rhythm is NOT normal sinus rhythm
    non_normal = ann[ann["rhythm_label"] != "N"]

    if len(non_normal) > 0:
        # value_counts() sorts by frequency descending; .index[0] is the most common value
        return non_normal["rhythm_label"].value_counts().index[0]
    else:
        # Every good-quality beat in this patient's file was normal sinus rhythm
        return "N"


# Sanity check on case 337
print("Case 337 dominant rhythm:", get_dominant_rhythm(337))

In [ ]:
# Apply the function to all 482 case IDs in our merged dataframe
merged["dominant_rhythm"] = merged["caseid"].apply(get_dominant_rhythm)

print("dominant_rhythm value counts across all 482 patients:")
print(merged["dominant_rhythm"].value_counts())
print()
print(f"Patients with no dominant rhythm available (None): {merged['dominant_rhythm'].isna().sum()}")

## Part D: Save the Updated Dataframe

In [ ]:
print(f"merged shape before saving: {merged.shape}")
assert merged["caseid"].is_unique, "caseid should still be unique after adding the new column"

merged.to_csv(MERGED_CSV, index=False)
print(f"Saved updated merged_482_cases.csv with dominant_rhythm column added: {merged.shape}")

## Part E: Filter Out Noise-Labeled Cases

5 of the 482 patients have `dominant_rhythm == "Noise"` (no patient in this cohort
has `"Unclassifiable"`). A Noise-dominant annotation means the rhythm label itself
isn't trustworthy for that patient, not that any physiological signal was observed —
these are dropped before imputation and feature extraction. Produces
`data/interim/rhythm_filtered_cases.csv` (477 rows), the input to
`03_handle_missing_values.ipynb`.

In [ ]:
RHYTHM_FILTERED_CSV = BASE_DIR / "data" / "interim" / "rhythm_filtered_cases.csv"

EXCLUDED_RHYTHMS = ["Noise", "Unclassifiable"]
n_before = len(merged)
filtered = merged[~merged["dominant_rhythm"].isin(EXCLUDED_RHYTHMS)].reset_index(drop=True)
n_dropped = n_before - len(filtered)

print(f"Dropped {n_dropped} patients with dominant_rhythm in {EXCLUDED_RHYTHMS}:")
print(merged[merged["dominant_rhythm"].isin(EXCLUDED_RHYTHMS)][["caseid", "dominant_rhythm"]].to_string(index=False))

filtered.to_csv(RHYTHM_FILTERED_CSV, index=False)
print(f"\\nSaved {filtered.shape[0]} rows x {filtered.shape[1]} columns to {RHYTHM_FILTERED_CSV}")